In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D,
    MaxPooling2D,
    BatchNormalization,
    Dropout,
    Flatten,
    Dense
)
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical

# ==========================
# Load Data
# ==========================

train_df = pd.read_csv(
    "/kaggle/input/competitions/digit-recognizer/train.csv"
)

test_df = pd.read_csv(
    "/kaggle/input/competitions/digit-recognizer/test.csv"
)

# ==========================
# Prepare Data
# ==========================

X_train = train_df.drop("label", axis=1).values
y_train = train_df["label"].values

X_test = test_df.values

# Normalize
X_train = X_train.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0

# Reshape for CNN
X_train = X_train.reshape(-1, 28, 28, 1)
X_test = X_test.reshape(-1, 28, 28, 1)

# One-hot encode labels
y_train = to_categorical(y_train, 10)

# ==========================
# Build CNN
# ==========================

model = Sequential([

    Conv2D(
        32,
        (3,3),
        activation='relu',
        input_shape=(28,28,1)
    ),
    BatchNormalization(),

    Conv2D(
        64,
        (3,3),
        activation='relu'
    ),
    BatchNormalization(),
    MaxPooling2D((2,2)),
    Dropout(0.25),

    Conv2D(
        128,
        (3,3),
        activation='relu'
    ),
    BatchNormalization(),
    MaxPooling2D((2,2)),
    Dropout(0.25),

    Flatten(),

    Dense(
        128,
        activation='relu'
    ),
    Dropout(0.5),

    Dense(
        10,
        activation='softmax'
    )
])

# ==========================
# Compile
# ==========================

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# ==========================
# Early Stopping
# ==========================

early_stop = EarlyStopping(
    monitor='val_accuracy',
    patience=3,
    restore_best_weights=True
)

# ==========================
# Train
# ==========================

history = model.fit(
    X_train,
    y_train,
    validation_split=0.1,
    epochs=20,
    batch_size=128,
    callbacks=[early_stop],
    verbose=1
)


# ==========================
# Predict Test Data
# ==========================

predictions = model.predict(X_test)

predicted_labels = np.argmax(
    predictions,
    axis=1
)

# ==========================
# Create Submission
# ==========================

submission = pd.read_csv(
    "/kaggle/input/competitions/digit-recognizer/sample_submission.csv"
)

submission["Label"] = predicted_labels

submission.to_csv(
    "submission.csv",
    index=False
)

print(submission.head())

print("\nSubmission file created successfully!")

# Verify files
import os
print("\nOutput Files:")
print(os.listdir("/kaggle/working"))